**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Adaptive Filtering: LMS to the Affine Projection Algorithm

In [Filter Design](../Intro_DSP/Filter_Design.ipynb) we designed filters *once*, by hand. But what if the interference drifts, the room echo changes, the channel fades? Then the filter must **redesign itself from the data, every sample**. This workshop builds that idea from steepest descent → LMS → NLMS → the Affine Projection Algorithm (APA).

## 0. Introduction

The canonical setup — *system identification*:

```
 x[n] ──► unknown system ──► d[n] (+ noise)
 x[n] ──► our filter w ────► y[n]
                 e[n] = d[n] − y[n]  drives the adaptation
```

The same loop, rewired, does echo cancellation, channel equalization, and noise cancellation — adaptive filtering is one algorithm wearing four costumes.

## 1. Pre-requisites

- [Foundations of Signal Processing](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — convolution, FIR structure.
- [Intro to Python](../Intro_Programming/Intro_Python/Intro_Python.ipynb) — NumPy.
- Comfort with gradients (any calculus course).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# The "unknown" system our filter must discover: a 16-tap FIR echo path
M = 16
w_true = np.exp(-0.4 * np.arange(M)) * np.cos(0.9 * np.arange(M))
w_true /= np.linalg.norm(w_true)

N = 4000                       # samples
x = rng.standard_normal(N)     # white input (we'll break this assumption later!)
d = np.convolve(x, w_true)[:N] + 0.01 * rng.standard_normal(N)

plt.figure(figsize=(7, 2.2))
plt.stem(w_true)
plt.title("The unknown system's impulse response (our target)")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1836971/4249226365.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 1 of 2 — *From Steepest Descent to LMS* (~35 min)
**Goal:** derive the Wiener solution, then strip it down to the LMS update and watch it converge.
**Feeds into:** Session 2 (NLMS & APA).

---

## 2. Theory: the Wiener Filter & LMS

💡 **Intuition.** Picture the error surface: for an FIR filter the mean-squared error is a **bowl** in weight space — one unique bottom (the Wiener solution). Steepest descent walks downhill using the *true* gradient, which needs statistics ($R$, $p$) we never have. LMS makes one audacious move: replace the expected gradient with its **one-sample estimate**. Each step is noisy, but the *average* direction is still downhill.

### 2.1. The Wiener Solution

Minimizing $J(\mathbf{w}) = E[e^2[n]]$ with $e[n] = d[n] - \mathbf{w}^T\mathbf{x}[n]$ gives

$$\nabla J = 2R\mathbf{w} - 2\mathbf{p} = 0 \;\Rightarrow\; \mathbf{w}_o = R^{-1}\mathbf{p}$$

with $R = E[\mathbf{x}\mathbf{x}^T]$ (input autocorrelation) and $\mathbf{p} = E[d\,\mathbf{x}]$ (cross-correlation). Two problems: we don't know $R$ and $\mathbf{p}$, and inverting $R$ is expensive. Adaptive filtering is the art of *approaching* $\mathbf{w}_o$ without ever forming it.

### 2.2. The LMS Update

Substitute the instantaneous gradient estimate $\hat{\nabla} J = -2 e[n]\, \mathbf{x}[n]$ into gradient descent:

$$\boxed{\;\mathbf{w}[n+1] = \mathbf{w}[n] + \mu\, e[n]\, \mathbf{x}[n]\;}$$

Three multiplies per tap per sample. Convergence requires $0 < \mu < \frac{2}{\lambda_{max}}$ (the largest eigenvalue of $R$) — step too far and the bowl becomes a trampoline.

In [2]:
def lms(x, d, M, mu):
    w = np.zeros(M)
    e = np.zeros(len(x))
    W = np.zeros((len(x), M))          # weight history, for plotting
    xbuf = np.zeros(M)
    for k in range(len(x)):
        xbuf = np.roll(xbuf, 1); xbuf[0] = x[k]
        y = w @ xbuf
        e[k] = d[k] - y
        w = w + mu * e[k] * xbuf
        W[k] = w
    return w, e, W

w_lms, e_lms, W_lms = lms(x, d, M, mu=0.02)
print("final weight error:", np.linalg.norm(w_lms - w_true).round(4))

final weight error: 0.0042


In [3]:
plt.figure(figsize=(8, 2.8))
plt.plot(10 * np.log10(np.convolve(e_lms**2, np.ones(50)/50, "valid") + 1e-12))
plt.title("LMS learning curve (smoothed |e|² in dB)")
plt.xlabel("sample"); plt.ylabel("MSE [dB]"); plt.grid(True)
plt.tight_layout(); plt.show()

/tmp/ipykernel_1836971/2248593106.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The error drops ~30+ dB and flattens at the noise floor: the filter has *discovered* the unknown system. Try `mu=0.2` — divergence is loud and immediate.

---
### 🕐 Session 2 of 2 — *NLMS & the Affine Projection Algorithm* (~40 min)
**Goal:** fix LMS's sensitivity to input power and correlation; implement NLMS and APA and compare all three.
**Builds on:** Session 1.

---

## 3. NLMS: Normalize the Step

💡 **Intuition.** LMS's step size is entangled with the input's *power*: loud input ⇒ effectively huge steps ⇒ instability. NLMS divides the step by $\|\mathbf{x}[n]\|^2$, so each update moves the weights *just enough to cancel the current error* — a projection onto the hyperplane of solutions for the newest sample, scaled by $\mu$.

$$\mathbf{w}[n+1] = \mathbf{w}[n] + \frac{\mu}{\epsilon + \|\mathbf{x}[n]\|^2}\, e[n]\, \mathbf{x}[n]$$

with small $\epsilon$ guarding the division. Stable for $0 < \mu < 2$ regardless of input power.

In [4]:
def nlms(x, d, M, mu, eps=1e-6):
    w = np.zeros(M); e = np.zeros(len(x)); xbuf = np.zeros(M)
    for k in range(len(x)):
        xbuf = np.roll(xbuf, 1); xbuf[0] = x[k]
        e[k] = d[k] - w @ xbuf
        w = w + (mu / (eps + xbuf @ xbuf)) * e[k] * xbuf
    return w, e

## 4. APA: Project Onto Many Constraints at Once

💡 **Intuition.** NLMS satisfies only the **newest** sample's equation, so with *correlated* input (speech, music — anything non-white) consecutive updates keep undoing each other, and convergence crawls. APA keeps the last $K$ input vectors and jumps to the nearest weight vector satisfying **all $K$ equations simultaneously** — an affine projection. $K=1$ recovers NLMS; larger $K$ churns through correlated input dramatically faster, at the price of a $K \times K$ solve per sample.

With $X_K[n] = [\mathbf{x}[n], \dots, \mathbf{x}[n-K+1]]$ (an $M \times K$ matrix) and $\mathbf{e}_K[n]$ the corresponding error vector:

$$\mathbf{w}[n+1] = \mathbf{w}[n] + \mu\, X_K (X_K^T X_K + \epsilon I)^{-1} \mathbf{e}_K[n]$$

In [5]:
def apa(x, d, M, mu, K, eps=1e-4):
    w = np.zeros(M); e = np.zeros(len(x)); xbuf = np.zeros(M)
    X = np.zeros((M, K)); dbuf = np.zeros(K)
    for k in range(len(x)):
        xbuf = np.roll(xbuf, 1); xbuf[0] = x[k]
        X = np.roll(X, 1, axis=1); X[:, 0] = xbuf
        dbuf = np.roll(dbuf, 1); dbuf[0] = d[k]
        eK = dbuf - X.T @ w
        e[k] = eK[0]
        w = w + mu * X @ np.linalg.solve(X.T @ X + eps * np.eye(K), eK)
    return w, e

### 4.1. The Fair Fight: Correlated Input

White input flatters every algorithm. Real signals are colored — so we generate a correlated input by low-pass filtering noise (an AR(1) process) and race all three.

In [6]:
from scipy import signal as sig

x_col = sig.lfilter([1.0], [1.0, -0.9], rng.standard_normal(N))   # heavily correlated
d_col = np.convolve(x_col, w_true)[:N] + 0.01 * rng.standard_normal(N)

_, e1 = nlms(x_col, d_col, M, mu=0.5)
w2, e2 = apa(x_col, d_col, M, mu=0.5, K=4)
w3, e3 = apa(x_col, d_col, M, mu=0.5, K=8)

def curve(e): return 10 * np.log10(np.convolve(e**2, np.ones(100)/100, "valid") + 1e-12)

plt.figure(figsize=(8, 3))
plt.plot(curve(e1), label="NLMS  (K=1)")
plt.plot(curve(e2), label="APA   K=4")
plt.plot(curve(e3), label="APA   K=8")
plt.legend(); plt.grid(True)
plt.title("Correlated input: projection order buys convergence speed")
plt.xlabel("sample"); plt.ylabel("MSE [dB]")
plt.tight_layout(); plt.show()

print("final weight error, APA K=8:", np.linalg.norm(w3 - w_true).round(4))

final weight error, APA K=8: 0.0126


/tmp/ipykernel_1836971/219609170.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Read the plot: same $\mu$, same data — but each increase in $K$ steepens the initial descent. The trade-offs to remember:

| | LMS | NLMS | APA-$K$ |
|---|---|---|---|
| Cost / sample | $O(M)$ | $O(M)$ | $O(K^2 M)$ |
| Robust to input power | ✗ | ✓ | ✓ |
| Fast on colored input | ✗ | ✗ | ✓ |
| Misadjustment (noise amp.) | low | med | grows with $K$ |

## 5. Conclusion

One loop — predict, err, correct — with increasingly clever corrections: LMS follows a noisy gradient, NLMS normalizes it, APA projects onto a window of constraints. The next step up that ladder replaces "window of constraints" with a full statistical model of the system's evolution — that is the Kalman filter.

---
## Where next

- [Adaptive Filtering: Kalman](./Intro_AdFilt_KF.ipynb) — the optimal recursive estimator.
- [Recurrent Neural Networks](./README.md#workshop-3--recurrent-neural-networks-available) — nonlinear models with internal state, same predict/correct heartbeat.
- [Filter Design](../Intro_DSP/Filter_Design.ipynb) — the fixed-filter baseline all of this improves upon.